In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
# Task 1: Write your code here:
path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path)

print(f"Shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
# Task 5: Write your code here:
# target distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis = 1)
df

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df.isnull().sum())
print("====================")
df['Weather'] = df['Weather'].fillna('none')
df['Traffic_Level'] = df['Traffic_Level'].fillna('none')
df['Time_of_Day'] = df['Time_of_Day'].fillna('none')
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df["Courier_Experience_yrs"].mean())
df['Delivery_Time'] = df['Delivery_Time'].fillna(df["Delivery_Time"].mean())
print("result")
print(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

df

In [ ]:
##DO ONEHOTENCODER IF YOU STILL HAVE TIME

# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:
#i am dealing with continuous data

In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']
X = np.array(X)
y = np.array(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
#We are dealing with continuous data, so i will use kfold
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=40, random_state=42, n_jobs=-1)

mae_scores = []
preds = []
for train_idx, val_idx in kfold.split(X):
    X_fold_train, X_fold_val = X[train_idx], X[val_idx]
    y_fold_train, y_fold_val = y[train_idx], y[val_idx]

    # Train and predict
    model.fit(X, y)
    y_fold_pred = model.predict(X_fold_val)
    for elem in y_fold_pred:
        preds.append(elem)
    # Calculate metrics
    mae = mean_absolute_error(y_fold_val, y_fold_pred)

    mae_scores.append(mae)


mae_scores = np.array(mae_scores)

print(f"10-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
average_losses = np.mean(mae_scores, axis=0)

plt.plot(mae_scores, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df.hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(pd.DataFrame(preds), "Sales")

In [ ]:
# Task Bonus: Write your code here: